# Header 1

## Header 2

### Header 3

#### Header 4

##### Header 5

###### Header 6

Test of normal Text

**Test of Bold Text**

*Test of Itallic Text*

# Getting Data Example

ODOT Plan Sets [via TIMS](https://gis.dot.state.oh.us/tims/map)

In [ ]:
import re
import json
import camelot
import requests
import numpy as np
from munch import Munch
from pathlib import Path
from bs4 import BeautifulSoup

In [ ]:
full_data_url = "https://gis.dot.state.oh.us/arcgis/rest/services/TIMS/Projects/MapServer/4/query?where=1%3D1&text=&objectIds=&time=&geometry=&geometryType=esriGeometryEnvelope&inSR=&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&returnTrueCurves=false&maxAllowableOffset=&geometryPrecision=&outSR=&having=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&outStatistics=&returnZ=false&returnM=false&gdbVersion=&historicMoment=&returnDistinctValues=false&resultOffset=&resultRecordCount=&queryByDistance=&returnExtentOnly=false&datumTransformation=&parameterValues=&rangeValues=&quantizationParameters=&featureEncoding=esriDefault&f=pjson"

page = requests.get(full_data_url)
data_json = json.loads(page.content)

In [ ]:
project_list = []

In [ ]:
for project in data_json["features"]:
    project_obj = Munch(project["attributes"])
    project_list.append(project_obj)

In [ ]:
# Pull this cookie
project_list[3].PROJECT_PLANS_URL

In [ ]:
headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Cookie": "_ga=GA1.3.1674095645.1710160295; _ga_H9WHB924EN=GS1.3.1710509539.2.0.1710509539.0.0.0; _gid=GA1.3.1622672811.1713537145; _ga_CCD0M9E21K=GS1.3.1713556609.7.1.1713556617.0.0.0; JSESSIONID=KPT7KjXS4CQVSIF8sbzXvCKgE5tMl6t9CD9WuU9l.dotidpxep02",
    "Host": "contracts.dot.state.oh.us",
    "Referer": "https://contracts.dot.state.oh.us/document/getDocumentProperties.do?documentId=974043&cabinetId=1002&context=documentSearch&tabId=-618cef00&displayMode=condensed&from=topNav&actionForm=documentSearchCriteriaForm&actionMode=display",
}

In [ ]:
len(project_list)

In [ ]:
for index, item in enumerate(project_list):
    if item.PID_NBR == 113626:
        print(index)
        break
else:
    index = -1

In [ ]:
project_list[0]

This next cell actually repeatedly calls the API, change it to a `code` cell and run it to begin downloading the data

# Splitting PDF Files into Constituent Pages

In [ ]:
import os
from pathlib import Path

plan_set_path = Path(
    "C:/Users/dane.parks/PycharmProjects/civilpy/Notebooks/output/ODOT_plans"
)
file_list = [plan_set_path / x for x in os.listdir(plan_set_path)]

In [ ]:
test_pdf = file_list[0]

In [ ]:
from pdf2image import convert_from_path

In [ ]:
pages = convert_from_path(test_pdf, 300)

In [ ]:
pages[0]

In [ ]:
path = Path(file_list[0])

path.parent / "img" / os.path.basename(file_list[0])[:-4]

In [ ]:
if not os.path.exists(path.parent / "img" / os.path.basename(file_list[0])[:-4]):
    print("Folder Doesn't Exist")
else:
    print("Folder Exists")

This next cell breaks the previously downloaded pdfs into individual `.jpg` files it takes a long time to run so is stored as a `raw` cell for this notebook. Convert it back to a `code` cell to make it run again.

# Reading a Table with Camelot

[Link to full Demo File](https://daneparks.com/Dane/civilpy/-/raw/master/Notebooks/res/100843.pdf?ref_type=heads)

[Link to Table Only Demo File](https://daneparks.com/Dane/civilpy/-/raw/master/Notebooks/res/100843%20-%20Table%20Only.pdf?ref_type=heads)

``` bash
# Installing Packages
$ pip install chardet

# Via Conda:
$ conda install -c conda-forge camelot-py
```

In [ ]:
path_to_pdf = Path(
    "C:/Users/dane.parks/PycharmProjects/civilpy/Notebooks/output/ODOT_plans/100843.pdf"
)

In [ ]:
tables = camelot.read_pdf(str(path_to_pdf), pages="5")

In [ ]:
tables

In [ ]:
tables[0].parsing_report

Notice that it shows the "accuracy" as being very high, but if we review what it's actually retrieved,

In [ ]:
tables[0].df

the results aren't that great. We can probably improve the accuracy by removing the areas of the sheet we don't want to process to improve our results, for instance here's the same process as above but ran on a "clean" version of the pdf with the column labels and title block removed.

In [ ]:
path_to_pdf_2 = (
    r"C:\Users\dane.parks\PycharmProjects\civilpy\Notebooks\res\100843 - Table Only.pdf"
)

In [ ]:
tables_2 = camelot.read_pdf(path_to_pdf_2)

In [ ]:
tables_2[0].parsing_report

In [ ]:
tables_2[0].df

We can verify what Camelot was interpreting:

In [ ]:
camelot.plot(tables_2[0], kind="text").show()

In [ ]:
import matplotlib.pyplot as pl

pl.show(camelot.plot(tables_2[0], kind="grid"))

Which Looks good, but it (correctly) included a lot of empty rows that aren't necessary for getting information from the table, and we have to provide the column labels, later we'll determine a way to automatically obtain these

In [ ]:
# Note: The first and last columns were dropped when moving over the table since they're blank, which is why '3' and 'SEE SHEET NO.' aren't included here
column_headers = [
    "4",
    "6",
    "7",
    "8",
    "9",
    "drop1",
    "drop2",
    "drop3",
    "drop4",
    "drop5",
    "01/STR/PV",
    "02/STR/BR",
    "ALT (X)",
    "ITEM",
    "ITEM EXT",
    "GRAND TOTAL",
    "UNIT",
    "DESCRIPTION",
]

In [ ]:
tables_2[0].df.columns = column_headers
tables_2[0].df.shape

Now remove empty rows,

In [ ]:
# Replace empty strings with NaN
tables_2[0].df = tables_2[0].df[tables_2[0].df.astype(bool)]

# Drop empty rows if all values are NaN
tables_2[0].df.dropna(axis=0, how="all", inplace=True)

tables_2[0].df.shape

Remove empty columns

In [ ]:
# Drop empty rows if all values are NaN
tables_2[0].df.dropna(axis=1, how="all", inplace=True)

tables_2[0].df.shape

View the updated dataframe,

In [ ]:
tables_2[0].df

**Identifying patterns:**

The `Description` column actually contains two types of data, when it's centered and there's no values to the left of it, it indicates a Category. When there are values to the left, it indicates that the values in that row belong to that sub-category.

The demo sheet contains the following main categories:

- Erosion Control
- Pavement
- Traffic Control
- Structure Repair (ADA-136.21.39 : SFN0103527)
- Maintenance of Traffic
- Incidentals

for simplicity we can filter by the `GRAND TOTAL` column, since it will pretty much always contain a value for subcategories:

In [ ]:
# Kind of started code golfing here, so will explain w/ comments: returns a list of categories
categories_list = (
    tables_2[0].df[tables_2[0].df["GRAND TOTAL"].isna()]["DESCRIPTION"].to_list()
)
categories_list

To get the `index` (Row #) where each value occurs we can do,

In [ ]:
# Returns the index for each value in the categories list
index_list = [
    tables_2[0].df.index.get_loc(
        tables_2[0].df[tables_2[0].df["DESCRIPTION"] == x].index[0]
    )
    for x in categories_list
]

In [ ]:
# Squishes the two lists above into a list of tuples with the index, category
combined_values = list(map(lambda x, y: (x, y), index_list, categories_list))
combined_values

Being a bridge guy, I'm most interested in the 3rd category, particularly in which bridge this plan set includes values for.

We can use regex to pull the SFN and CTY-RTE-SEC out of the string value, which then identifies which structure this plan set is related to.

In [ ]:
import re

# CTY-RTE-SEC: 3 uppercase letters, hyphen, route digits, hyphen, section with decimal
# RTE typically all digits (e.g. 071); section is decimal (e.g. 2.87)
# Escaped decimal \. and + for one-or-more digits on each side
cty_rte_sec_ptn = re.compile(r"([A-Z]{3})-(\d+)-(\d+\.\d+)")

# SFN: always exactly 7 digits; "SFN " prefix is optional in free-text
sfn_ptn = re.compile(r"(SFN )?(\d{7})")


with these patterns we can take the string,

In [ ]:
combined_values[3][1]

and get,

In [ ]:
cty_rte_sec = re.search(cty_rte_sec_ptn, combined_values[3][1]).group(0)
sfn = re.search(sfn_ptn, combined_values[3][1]).group(2)
print(f"{cty_rte_sec=:}")
print(f"{sfn=:}")

Because of how the regex patterns are structured we can also get,

In [ ]:
cte = cty_rte_sec = re.search(cty_rte_sec_ptn, combined_values[3][1]).group(1)
rte = cty_rte_sec = re.search(cty_rte_sec_ptn, combined_values[3][1]).group(2)
sec = cty_rte_sec = re.search(cty_rte_sec_ptn, combined_values[3][1]).group(3)
print(f"{cte=}\n{rte=:}\n{sec=:}")

or,

In [ ]:
sfn = re.search(sfn_ptn, combined_values[3][1]).group(0)
print(f"{sfn=:}")

# Finding the Correct Pages to Read

Camelot has two major caveats. The first being it only works on text based tables (So scanned/image pdfs won't work). The Second being what was demonstrated earlier where reading full pages often results in inaccurate results because of Camelot attempting to read the title block.

In order to avoid the process of clipping out the tables of every pdf, which would be extremely tedious, we can attempt to generate bounding boxes of the values with AI processes known as Image Localization, Object Detection and Image Segmentation.

## Reading Image Tables

[tablecv](https://pypi.org/project/tablecv/)

Generally considered one of the better "Image based table" converters, provides lackluster results on older handwritten tables that are scanned in.

In [ ]:
from tablecv import extract_table

In [ ]:
image_pdf_path = Path(
    r"C:\Users\dane.parks\PycharmProjects\civilpy\Notebooks\res\test_image_pdf.pdf"
)

In [ ]:
pages = convert_from_path(image_pdf_path, 300)

In [ ]:
pages[0].save("test_image_pdf.jpg")

Note that tableCV doesn't work when the page consists of a lot of details outside the table

In [ ]:
df = extract_table(image_path="test_image_pdf.jpg")

In [ ]:
table_pdf_path = Path(
    r"C:\Users\dparks1\Title Sheets\1997\1997_ASD_sht-001.tif"
)

In [ ]:
pages = convert_from_path(table_pdf_path, 300)

In [ ]:
pages[0].save("table_image_pdf.jpg")

And provides pretty spotty results even when a table is isolated

In [ ]:
df = extract_table(image_path="table_image_pdf.jpg")

In [ ]:
df

In [ ]:
table_pdf_path = Path(
    r"C:\Users\dane.parks\PycharmProjects\civilpy\Notebooks\res\test_image_pdf_single_table_computer.pdf"
)

In [ ]:
pages = convert_from_path(table_pdf_path, 300)

In [ ]:
pages[0].save("computer_image_pdf.jpg")

Even with computer generated tables, the tablecv library demonstrates low accuracy performance,

In [ ]:
df = extract_table(image_path="computer_image_pdf.jpg")

# Customized AI Models for Extracting Useful Features from Documents

## Extraction Phase

In [2]:
import os
import glob
import logging
from PIL import Image
from pathlib import Path
from src.civilpy.state.ohio.DOT.title_sheet import load_trained_model, export_for_review

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding='utf-8'), # Handler for the file
        logging.StreamHandler()             # Handler for the console/Jupyter output
    ]
)

# --- DISABLE DECOMPRESSION BOMB CHECK ---
Image.MAX_IMAGE_PIXELS = None
# ----------------------------------------


def main():
    """
    Loads the model once, then loops through input files to extract data
    for manual review, with all output captured by the logging setup.
    """
    logging.info("--- Starting Extraction Process ---")

    file_types = ('*.pdf', '*.png', '*.jpg', '*.jpeg', '*.tiff', '*.tif')
    input_files = []
    for ext in file_types:
        input_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))

    if not input_files:
        logging.warning(f"No files found in '{INPUT_DIR}'. Please check the path.")
        return

    try:
        model = load_trained_model(MODEL_PATH)
    except FileNotFoundError as e:
        logging.error(e)
        return
        
    successful_extractions = 0
    for file_path in input_files:
        success = extract_data_for_review(
            model=model,
            file_path=file_path,
            review_dir=REVIEW_DIR,
            target_label=TARGET_LABEL,
            page_num=1
        )
        if success:
            successful_extractions += 1
            
    logging.info("--- Extraction Complete ---")
    logging.info(f"Successfully processed {successful_extractions} out of {len(input_files)} files.")
    logging.info(f"Log file saved to '{log_file_name}'.")

In [ ]:
for i in range(1996, 2025, 1):
    print(i)
    
    # --- Script CONFIGURATION ---
    INPUT_DIR = Path(f"C:\\Users\\dparks1\\Title Sheets\\{i}")
    REVIEW_DIR = INPUT_DIR / "for_manual_review"
    MODEL_PATH = "trained_model.pth"
    TARGET_LABEL = "Standard Construction Drawings" 
    log_file_name = REVIEW_DIR / "_script_output.txt"
    
    # Clear the log file if it exists from a previous run
    if not os.path.exists(REVIEW_DIR):
        os.mkdir(REVIEW_DIR)
    
    # Clear the log file if it exists from a previous run
    if os.path.exists(log_file_name):
        os.remove(log_file_name)
    
    main()

## Review Finalization Phase

After the excel sheets have been finalized, write the recorded data to json,

In [ ]:
import os
import glob
from pathlib import Path
from src.civilpy.state.ohio.DOT.title_sheet import finalize_cleaned_data

In [ ]:
# --- Script CONFIGURATION ---
INPUT_DIR = Path(r"C:\Users\dparks1\Title Sheets\1996")
REVIEW_DIR = INPUT_DIR / "for_manual_review"
MODEL_PATH = "trained_model.pth"
TARGET_LABEL = "Standard Construction Drawings" 
log_file_name = REVIEW_DIR / "_script_output.txt"
FINAL_OUTPUT_DIR = REVIEW_DIR / "final_json_output"

In [ ]:
def main():
    """
    Loops through cleaned Excel files and converts them to final JSON output.
    """
    print("--- Starting Finalization Process ---")
    
    # Find all Excel files in the review directory
    cleaned_files = glob.glob(os.path.join(REVIEW_DIR, '*.xlsx'))
    
    if not cleaned_files:
        print(f"No cleaned Excel files (.xlsx) found in '{REVIEW_DIR}'.")
        return

    successful_finalizations = 0
    # Loop through each cleaned file and process it
    for file_path in cleaned_files:
        success = finalize_cleaned_data(
            cleaned_excel_path=file_path,
            final_dir=FINAL_OUTPUT_DIR
        )
        if success:
            successful_finalizations += 1

    print("\n--- Finalization Complete ---")
    print(f"Successfully finalized {successful_finalizations} out of {len(cleaned_files)} files.")
    print(f"Check for final output in the '{FINAL_OUTPUT_DIR}' directory.")

In [ ]:
if __name__ == "__main__":
    main()

# Appendix A - Demos

The following examples are from TensorFlow's documentation, they demonstrate various principles, [Image Classification](#ImageClassification), [Text Classification](#TextClassification), [Imge Localization](#ImageLocalization), [Object Detection](#ObjectDetection), and [Image Segmentation](#ImageSegmentation)

<a id='ImageClassification'></a>
## Image Classification Model using TensorFlow

Quick tutorial following [this guide.](https://www.tensorflow.org/tutorials/keras/classification) Uses the "Fashion-MNIST" as an example problem to solve to verify model is being setup and run correcly.

In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [ ]:
# Training sets
train_images, train_labels

# Testing sets
test_images, test_labels

# To prevent output
print()

the images are 28x28 numpy arrays (a value for each pixel). with pixel values ranging from 0 to 255, the labels are arrays of integers, ranging from 0-9, these correspond to the *class* of clothing the image represents:

| Label |     Class       |
|:-----:|:---------------:|
| 0     |     T-shirt/top |
| 1     |     Trouser     |
| 2     |     Pullover    |
| 3     |     Dress       |
| 4     |     Coat        |
| 5     |     Sandal      |
| 6     |     Shirt       |
| 7     |     Sneaker     |
| 8     |     Bag         |
| 9     |     Ankle boot  |

In [ ]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

### Explore the data

Let's explore the format of the dataset before training the model. The following shows there are 60,000 images in the training set, with each image represented as 28 x 28 pixels:

In [ ]:
train_images.shape

In [ ]:
len(train_labels)

### Preprocess the data

The data must be preprocessed before training the network. If you inspect the first image in the training set, you will see that the pixel values fall in the range of 0 to 255:

In [ ]:
plt.figure()
plt.imshow(train_images[0])
plt.colorbar()
plt.grid(False)
plt.show()

Scale these values to a range of 0 to 1 before feeding them to the neural network model. To do so, divide the values by 255. It's important that the training set and the testing set be preprocessed in the same way:

In [ ]:
train_images = train_images / 255.0

test_images = test_images / 255.0

To verify that the data is in the correct format and that you're ready to build and train the network, let's display the first 25 images from the training set and display the class name below each image.

In [ ]:
plt.figure(figsize=(10, 10))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])

plt.show()

### Build the model

Building the neural network requires configuring the layers of the model, then compiling the model.

Set up the layers
The basic building block of a neural network is the layer. Layers extract representations from the data fed into them. Hopefully, these representations are meaningful for the problem at hand.

Most of deep learning consists of chaining together simple layers. Most layers, such as tf.keras.layers.Dense, have parameters that are learned during training.

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10),
    ]
)

The first layer in this network, `tf.keras.layers.Flatten`, transforms the format of the images from a two-dimensional array (of 28 by 28 pixels) to a one-dimensional array (of 28 * 28 = 784 pixels). Think of this layer as unstacking rows of pixels in the image and lining them up. This layer has no parameters to learn; it only reformats the data.

After the pixels are flattened, the network consists of a sequence of two tf.keras.layers.Dense layers. These are densely connected, or fully connected, neural layers. The first Dense layer has 128 nodes (or neurons). The second (and last) layer returns a logits array with length of 10. Each node contains a score that indicates the current image belongs to one of the 10 classes.

### Compile the model

Before the model is ready for training, it needs a few more settings. These are added during the model's compile step:

- *Optimizer* — This is how the model is updated based on the data it sees and its loss function.
- *Loss function* — This measures how accurate the model is during training. You want to minimize this function to "steer" the model in the right direction.
- *Metrics* — Used to monitor the training and testing steps. The following example uses accuracy, the fraction of the images that are correctly classified.

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

### Train the model

Training the neural network model requires the following steps:

1. Feed the training data to the model. In this example, the training data is in the train_images and train_labels arrays.
1. The model learns to associate images and labels.
1. You ask the model to make predictions about a test set—in this example, the test_images array.
1. Verify that the predictions match the labels from the test_labels array.

### Feed the model
To start training, call the model.fit method—so called because it "fits" the model to the training data:

In [ ]:
model.fit(train_images, train_labels, epochs=10)

As the model trains, the loss and accuracy metrics are displayed. This model reaches an accuracy of about 0.91 (or 91%) on the training data.

### Evaluate accuracy

Next, compare how the model performs on the test dataset:

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)

print("\nTest accuracy:", test_acc)

It turns out that the accuracy on the test dataset is a little less than the accuracy on the training dataset. This gap between training accuracy and test accuracy represents overfitting. Overfitting happens when a machine learning model performs worse on new, previously unseen inputs than it does on the training data. An overfitted model "memorizes" the noise and details in the training dataset to a point where it negatively impacts the performance of the model on the new data. For more information, see the following:

- [Demonstrate overfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit#demonstrate_overfitting)
- [Strategies to prevent overfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit#strategies_to_prevent_overfitting)

### Make Predictions

With the model trained, you can use it to make predictions about some images. Attach a softmax layer to convert the model's linear outputs—[logits](https://developers.google.com/machine-learning/glossary#logits)—to probabilities, which should be easier to interpret.

In [ ]:
probability_model = tf.keras.Sequential([model, tf.keras.layers.Softmax()])

In [ ]:
predictions = probability_model.predict(test_images)

Here, the model has predicted the label for each image in the testing set. Let's take a look at the first prediction:

In [ ]:
predictions[0]

A prediction is an array of 10 numbers. They represent the model's "confidence" that the image corresponds to each of the 10 different articles of clothing. You can see which label has the highest confidence value:

In [ ]:
np.argmax(predictions[0])

So, the model is most confident that this image is an ankle boot, or class_names[9]. Examining the test label shows that this classification is correct:

In [ ]:
test_labels[0]

Define functions to graph the full set of 10 class predictions.

In [ ]:
def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    plt.imshow(img, cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = "blue"
    else:
        color = "red"

    plt.xlabel(
        "{} {:2.0f}% ({})".format(
            class_names[predicted_label],
            100 * np.max(predictions_array),
            class_names[true_label],
        ),
        color=color,
    )


def plot_value_array(i, predictions_array, true_label):
    true_label = true_label[i]
    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    thisplot = plt.bar(range(10), predictions_array, color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)

    thisplot[predicted_label].set_color("red")
    thisplot[true_label].set_color("blue")

### Verify predictions

With the model trained, you can use it to make predictions about some images.

Let's look at the 0th image, predictions, and prediction array. Correct prediction labels are blue and incorrect prediction labels are red. The number gives the percentage (out of 100) for the predicted label.

In [ ]:
i = 0
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plot_image(i, predictions[i], test_labels, test_images)
plt.subplot(1, 2, 2)
plot_value_array(i, predictions[i], test_labels)
plt.show()

In [ ]:
i = 12
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plot_image(i, predictions[i], test_labels, test_images)
plt.subplot(1, 2, 2)
plot_value_array(i, predictions[i], test_labels)
plt.show()

Let's plot several images with their predictions. Note that the model can be wrong even when very confident.

In [ ]:
# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 5
num_cols = 3
num_images = num_rows * num_cols
plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i, predictions[i], test_labels, test_images)
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 2)
    plot_value_array(i, predictions[i], test_labels)
plt.tight_layout()
plt.show()

### Use the trained model

Finally, use the trained model to make a prediction about a single image.

In [ ]:
# Grab an image from the test dataset.
img = test_images[1]

print(img.shape)

`tf.keras` models are optimized to make predictions on a batch, or collection, of examples at once. Accordingly, even though you're using a single image, you need to add it to a list:

In [ ]:
# Add the image to a batch where it's the only member.
img = np.expand_dims(img, 0)

print(img.shape)

Now predict the correct label for this image:

In [ ]:
predictions_single = probability_model.predict(img)

print(predictions_single)

In [ ]:
plot_value_array(1, predictions_single[0], test_labels)
_ = plt.xticks(range(10), class_names, rotation=45)
plt.show()

`tf.keras.Model.predict` returns a list of lists—one list for each image in the batch of data. Grab the predictions for our (only) image in the batch:

In [ ]:
np.argmax(predictions_single[0])

<a id='TextClassification'></a>

## Basic Text Classification

[Source](https://www.tensorflow.org/tutorials/keras/text_classification)

In [ ]:
import matplotlib.pyplot as plt
import os
import re
import shutil
import string
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import losses

In [ ]:
print(tf.__version__)

### Sentiment analysis

**Useful for Bridge Inspection Comments**

This notebook trains a sentiment analysis model to classify movie reviews as positive or negative, based on the text of the review. This is an example of binary—or two-class—classification, an important and widely applicable kind of machine learning problem.

You'll use the [Large Movie Review Dataset](https://ai.stanford.edu/%7Eamaas/data/sentiment/) that contains the text of 50,000 movie reviews from the [Internet Movie Database](https://www.imdb.com/). These are split into 25,000 reviews for training and 25,000 reviews for testing. The training and testing sets are balanced, meaning they contain an equal number of positive and negative reviews.

### Download and explore the IMDB dataset

Let's download and extract the dataset, then explore the directory structure.

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file(
    "aclImdb_v1", url, untar=True, cache_dir=".", cache_subdir=""
)

dataset_dir = os.path.join(os.path.dirname(dataset), "aclImdb")

In [ ]:
os.listdir(dataset_dir)

In [ ]:
train_dir = os.path.join(dataset_dir, "train")
os.listdir(train_dir)

The aclImdb/train/pos and aclImdb/train/neg directories contain many text files, each of which is a single movie review. Let's take a look at one of them.

In [ ]:
sample_file = os.path.join(train_dir, "pos/1181_9.txt")
with open(sample_file) as f:
    print(f.read())

### Load the dataset

Next, you will load the data off disk and prepare it into a format suitable for training. To do so, you will use the helpful text_dataset_from_directory utility, which expects a directory structure as follows.

To prepare a dataset for binary classification, you will need two folders on disk, corresponding to `class_a` and `class_b`. These will be the positive and negative movie reviews, which can be found in `aclImdb/train/pos` and `aclImdb/train/neg`. As the IMDB dataset contains additional folders, you will remove them before using this utility.

In [ ]:
remove_dir = os.path.join(train_dir, "unsup")
shutil.rmtree(remove_dir)

Next, you will use the `text_dataset_from_directory` utility to create a labeled `tf.data.Dataset`. tf.data is a powerful collection of tools for working with data.

When running a machine learning experiment, it is a best practice to divide your dataset into three splits: [train](https://developers.google.com/machine-learning/glossary#training_set), [validation](https://developers.google.com/machine-learning/glossary#validation_set), and [test](https://developers.google.com/machine-learning/glossary#test-set).

The IMDB dataset has already been divided into train and test, but it lacks a validation set. Let's create a validation set using an 80:20 split of the training data by using the `validation_split` argument below.

In [ ]:
batch_size = 32
seed = 42

raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed,
)

As you can see above, there are 25,000 examples in the training folder, of which you will use 80% (or 20,000) for training. As you will see in a moment, you can train a model by passing a dataset directly to `model.fit`. If you're new to [`tf.data`](https://www.tensorflow.org/api_docs/python/tf/data), you can also iterate over the dataset and print out a few examples as follows.

In [ ]:
for text_batch, label_batch in raw_train_ds.take(1):
    for i in range(3):
        print("Review", text_batch.numpy()[i])
        print("Label", label_batch.numpy()[i])

Notice the reviews contain raw text (with punctuation and occasional HTML tags like `<br/>`). You will show how to handle these in the following section.

The labels are 0 or 1. To see which of these correspond to positive and negative movie reviews, you can check the `class_names` property on the dataset.

In [ ]:
print("Label 0 corresponds to", raw_train_ds.class_names[0])
print("Label 1 corresponds to", raw_train_ds.class_names[1])

Next, you will create a validation and test dataset. You will use the remaining 5,000 reviews from the training set for validation.

<div class="alert alert-block alert-info">
    
**Note**: When using the validation_split and subset arguments, make sure to either specify a random seed, or to pass shuffle=False, so that the validation and training splits have no overlap.
</div>

In [ ]:
raw_val_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed,
)

In [ ]:
raw_test_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size
)

### Prepare the dataset for training

Next, you will standardize, tokenize, and vectorize the data using the helpful `tf.keras.layers.TextVectorization` layer.

Standardization refers to preprocessing the text, typically to remove punctuation or HTML elements to simplify the dataset. Tokenization refers to splitting strings into tokens (for example, splitting a sentence into individual words, by splitting on whitespace). Vectorization refers to converting tokens into numbers so they can be fed into a neural network. All of these tasks can be accomplished with this layer.

As you saw above, the reviews contain various HTML tags like `<br />`. These tags will not be removed by the default standardizer in the `TextVectorization` layer (which converts text to lowercase and strips punctuation by default, but doesn't strip HTML). You will write a custom standardization function to remove the HTML.

<div class="alert alert-block alert-info">

**Note**: To prevent training-testing skew (also known as training-serving skew), it is important to preprocess the data identically at train and test time. To facilitate this, the TextVectorization layer can be included directly inside your model, as shown later in this tutorial.
    
</div>

In [ ]:
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape(string.punctuation), ""
    )

Next, you will create a `TextVectorization` layer. You will use this layer to standardize, tokenize, and vectorize our data. You set the `output_mode` to `int` to create unique integer indices for each token.

Note that you're using the default split function, and the custom standardization function you defined above. You'll also define some constants for the model, like an explicit maximum `sequence_length`, which will cause the layer to pad or truncate sequences to exactly `sequence_length` values.

In [ ]:
max_features = 10000
sequence_length = 250

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=max_features,
    output_mode="int",
    output_sequence_length=sequence_length,
)

Next, you will call adapt to fit the state of the preprocessing layer to the dataset. This will cause the model to build an index of strings to integers.

<div class="alert alert-block alert-info">

**Note**: It's important to only use your training data when calling adapt (using the test set would leak information).
    
</div>

In [ ]:
# Make a text-only dataset (without labels), then call adapt
train_text = raw_train_ds.map(lambda x, y: x)
vectorize_layer.adapt(train_text)

Let's create a function to see the result of using this layer to preprocess some data.

In [ ]:
def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    return vectorize_layer(text), label

In [ ]:
# retrieve a batch (of 32 reviews and labels) from the dataset
text_batch, label_batch = next(iter(raw_train_ds))
first_review, first_label = text_batch[0], label_batch[0]
print("Review", first_review)
print("Label", raw_train_ds.class_names[first_label])
print("Vectorized review", vectorize_text(first_review, first_label))

As you can see above, each token has been replaced by an integer. You can lookup the token (string) that each integer corresponds to by calling .get_vocabulary() on the layer.

In [ ]:
print("1287 ---> ", vectorize_layer.get_vocabulary()[1287])
print(" 313 ---> ", vectorize_layer.get_vocabulary()[313])
print("Vocabulary size: {}".format(len(vectorize_layer.get_vocabulary())))

You are nearly ready to train your model. As a final preprocessing step, you will apply the TextVectorization layer you created earlier to the train, validation, and test dataset.

In [ ]:
train_ds = raw_train_ds.map(vectorize_text)
val_ds = raw_val_ds.map(vectorize_text)
test_ds = raw_test_ds.map(vectorize_text)

### Configure the dataset for performance

These are two important methods you should use when loading data to make sure that I/O does not become blocking.

`.cache()` keeps data in memory after it's loaded off disk. This will ensure the dataset does not become a bottleneck while training your model. If your dataset is too large to fit into memory, you can also use this method to create a performant on-disk cache, which is more efficient to read than many small files.

`.prefetch()` overlaps data preprocessing and model execution while training.

You can learn more about both methods, as well as how to cache data to disk in the [data performance guide.](https://www.tensorflow.org/guide/data_performance)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Create the model

It's time to create your neural network:

In [ ]:
embedding_dim = 16

In [ ]:
model = tf.keras.Sequential(
    [
        layers.Embedding(max_features, embedding_dim),
        layers.Dropout(0.2),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ]
)

model.summary()

The layers are stacked sequentially to build the classifier:

1. The first layer is an Embedding layer. This layer takes the integer-encoded reviews and looks up an embedding vector for each word-index. These vectors are learned as the model trains. The vectors add a dimension to the output array. The resulting dimensions are: (batch, sequence, embedding). To learn more about embeddings, check out the Word embeddings tutorial.
1. Next, a GlobalAveragePooling1D layer returns a fixed-length output vector for each example by averaging over the sequence dimension. This allows the model to handle input of variable length, in the simplest way possible.
1. The last layer is densely connected with a single output node.


### Loss function and optimizer

A model needs a loss function and an optimizer for training. Since this is a binary classification problem and the model outputs a probability (a single-unit layer with a sigmoid activation), you'll use losses.BinaryCrossentropy loss function.

Now, configure the model to use an optimizer and a loss function:

In [ ]:
model.compile(
    loss=losses.BinaryCrossentropy(),
    optimizer="adam",
    metrics=[tf.metrics.BinaryAccuracy(threshold=0.5)],
)

### Train the model

You will train the model by passing the dataset object to the fit method.

In [ ]:
epochs = 10
history = model.fit(train_ds, validation_data=val_ds, epochs=epochs)

### Evaluate the model

Let's see how the model performs. Two values will be returned. Loss (a number which represents our error, lower values are better), and accuracy.

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Loss: ", loss)
print("Accuracy: ", accuracy)

This fairly naive approach achieves an accuracy of about 86%.

### Create a plot of accuracy and loss over time

model.fit() returns a History object that contains a dictionary with everything that happened during training:

In [ ]:
history_dict = history.history
history_dict.keys()

There are four entries: one for each monitored metric during training and validation. You can use these to plot the training and validation loss for comparison, as well as the training and validation accuracy:

In [ ]:
acc = history_dict["binary_accuracy"]
val_acc = history_dict["val_binary_accuracy"]
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]

epochs = range(1, len(acc) + 1)

# "bo" is for "blue dot"
plt.plot(epochs, loss, "bo", label="Training loss")
# b is for "solid blue line"
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")

plt.show()

In this plot, the dots represent the training loss and accuracy, and the solid lines are the validation loss and accuracy.

Notice the training loss decreases with each epoch and the training accuracy increases with each epoch. This is expected when using a gradient descent optimization—it should minimize the desired quantity on every iteration.

This isn't the case for the validation loss and accuracy—they seem to peak before the training accuracy. This is an example of overfitting: the model performs better on the training data than it does on data it has never seen before. After this point, the model over-optimizes and learns representations specific to the training data that do not generalize to test data.

For this particular case, you could prevent overfitting by simply stopping the training when the validation accuracy is no longer increasing. One way to do so is to use the tf.keras.callbacks.EarlyStopping callback.

### Export the model

In the code above, you applied the TextVectorization layer to the dataset before feeding text to the model. If you want to make your model capable of processing raw strings (for example, to simplify deploying it), you can include the TextVectorization layer inside your model. To do so, you can create a new model using the weights you just trained.

In [ ]:
export_model = tf.keras.Sequential(
    [vectorize_layer, model, layers.Activation("sigmoid")]
)

export_model.compile(
    loss=losses.BinaryCrossentropy(from_logits=False),
    optimizer="adam",
    metrics=["accuracy"],
)

# Test it with `raw_test_ds`, which yields raw strings
loss, accuracy = export_model.evaluate(raw_test_ds)
print(accuracy)

### Inference on new data

To get predictions for new examples, you can simply call model.predict().

In [ ]:
examples = tf.constant(
    ["The movie was great!", "The movie was okay.", "The movie was terrible..."]
)

export_model.predict(examples)

Including the text preprocessing logic inside your model enables you to export a model for production that simplifies deployment, and reduces the potential for train/test skew.

There is a performance difference to keep in mind when choosing where to apply your TextVectorization layer. Using it outside of your model enables you to do asynchronous CPU processing and buffering of your data when training on GPU. So, if you're training your model on the GPU, you probably want to go with this option to get the best performance while developing your model, then switch to including the TextVectorization layer inside your model when you're ready to prepare for deployment.

Visit this tutorial to learn more about saving models.

### Exercise: multi-class classification on Stack Overflow questions

This tutorial showed how to train a binary classifier from scratch on the IMDB dataset. As an exercise, you can modify this notebook to train a multi-class classifier to predict the tag of a programming question on Stack Overflow.

A dataset has been prepared for you to use containing the body of several thousand programming questions (for example, "How can I sort a dictionary by value in Python?") posted to Stack Overflow. Each of these is labeled with exactly one tag (either Python, CSharp, JavaScript, or Java). Your task is to take a question as input, and predict the appropriate tag, in this case, Python.

The dataset you will work with contains several thousand questions extracted from the much larger public Stack Overflow dataset on BigQuery, which contains more than 17 million posts.

After downloading the dataset, you will find it has a similar directory structure to the IMDB dataset you worked with previously:

<div class="alert alert-block alert-info">

**Note**: To increase the difficulty of the classification problem, occurrences of the words Python, CSharp, JavaScript, or Java in the programming questions have been replaced with the word blank (as many questions contain the language they're about).
    
</div>

To complete this exercise, you should modify this notebook to work with the Stack Overflow dataset by making the following modifications:

1. At the top of your notebook, update the code that downloads the IMDB dataset with code to download the [Stack Overflow dataset](https://storage.googleapis.com/download.tensorflow.org/data/stack_overflow_16k.tar.gz) that has already been prepared. As the Stack Overflow dataset has a similar directory structure, you will not need to make many modifications.

1. Modify the last layer of your model to `Dense(4)`, as there are now four output classes.

1. When compiling the model, change the loss to [`tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`](https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy). This is the correct loss function to use for a multi-class classification problem, when the labels for each class are integers (in this case, they can be 0, 1, 2, or 3). In addition, change the metrics to `metrics=['accuracy']`, since this is a multi-class classification problem (`tf.metrics.BinaryAccuracy` is only used for binary classifiers).

1. When plotting accuracy over time, change `binary_accuracy` and `val_binary_accuracy` to `accuracy` and `val_accuracy`, respectively.

1. Once these changes are complete, you will be able to train a multi-class classifier.

### Learning more

This tutorial introduced text classification from scratch. To learn more about the text classification workflow in general, check out the [Text classification guide](https://developers.google.com/machine-learning/guides/text-classification/) from Google Developers.

<a id='ImageLocalization'></a>

## Image Localization

[Source](https://medium.com/analytics-vidhya/object-localization-using-keras-d78d6810d0be)

### Synthetic Usecase:

first of all, we will import and define the followings:

In [ ]:
import tensorflow as tf
from matplotlib import pyplot as plt
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
import numpy as np
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt

BATCH_SIZE = 64
EPOCH_SIZE = 64

Then, we will define our CNN based model. We will perform transfer learning: using a pre-trained version of VGG.

In [ ]:
# transfer learning - load pre-trained vgg and replace its head
vgg = tf.keras.applications.VGG16(
    input_shape=[128, 128, 3], include_top=False, weights="imagenet"
)
x = Flatten()(vgg.output)
x = Dense(3, activation="sigmoid")(x)
model1 = Model(vgg.input, x)
model1.compile(loss="binary_crossentropy", optimizer=Adam(lr=0.001))
# plot the model
plot_model(model1, "first_model.png", show_shapes=True, expand_nested=False)

Our next step will be creating a dataset using a python generator. It will create batches of white blobs over black background.

In [ ]:
from matplotlib.patches import Circle


def synthetic_gen(batch_size=64):
    # enable generating infinite amount of batches
    while True:
        # generate black images in the wanted size
        X = np.zeros((batch_size, 128, 128, 3))
        Y = np.zeros((batch_size, 3))
        # fill each image
        for i in range(batch_size):
            x = np.random.randint(8, 120)
            y = np.random.randint(8, 120)
            a = min(128 - max(x, y), min(x, y))
            r = np.random.randint(4, a)
            for x_i in range(128):
                for y_i in range(128):
                    if ((x_i - x) ** 2) + ((y_i - y) ** 2) < r**2:
                        X[i, x_i, y_i, :] = 1
            Y[i, 0] = (x - r) / 128.0
            Y[i, 1] = (y - r) / 128.0
            Y[i, 2] = 2 * r / 128.0
        yield X, Y


# sanity check - plot the images
x, y = next(synthetic_gen())
plt.imshow(x[0])

X is a batch of images and Y is the ground-truth bounding boxes. for each image, we will create a white blob, and we will make sure that it will be inside the borders of the image.

Now, we can train the model to predict the ground truth bounding-box, and visualize its performance:

In [ ]:
# needs steps per epoch since the generator is infinite
model1.fit_generator(synthetic_gen(), steps_per_epoch=EPOCH_SIZE, epochs=5)

In [ ]:
from matplotlib.patches import Rectangle


# given image and a label, plots the image + rectangle
def plot_pred(img, p):
    fig, ax = plt.subplots(1)
    ax.imshow(img)
    rect = Rectangle(
        xy=(p[1] * 128, p[0] * 128),
        width=p[2] * 128,
        height=p[2] * 128,
        linewidth=1,
        edgecolor="g",
        facecolor="none",
    )
    ax.add_patch(rect)
    plt.show()


# generate new image
x, _ = next(synthetic_gen())
# predict
pred = model1.predict(x)
# examine 1 image
im = x[0]
p = pred[0]
plot_pred(im, p)

### Semi-synthetic usecase:

For this usecase, we will use the same architecture, with a different type of images - images of a cat over a black background. This will be achieved using a different generator:

In [ ]:
from PIL import Image
from matplotlib.patches import Circle

cat_pil = Image.open("res/imgs/cat.png")
cat_pil = cat_pil.resize((64, 64))
cat = np.asarray(cat_pil)


def cat_gen(batch_size=64):
    # enable generating infinite amount of batches
    while True:
        # generate black images in the wanted size
        X = np.zeros((batch_size, 128, 128, 3))
        Y = np.zeros((batch_size, 3))
        # fill each image
        for i in range(batch_size):
            # resize the cat
            size = np.random.randint(32, 64)
            temp_cat = cat_pil.resize((size, size))
            cat = np.asarray(temp_cat) / 255.0
            cat_x, cat_y, _ = cat.shape
            # create a blank background image
            bg = Image.new("RGB", (128, 128))
            # generate
            x1 = np.random.randint(1, 128 - cat_x)
            y1 = np.random.randint(1, 128 - cat_y)
            # paste the cat over the image
            bg.paste(temp_cat, (x1, y1))
            cat = np.asarray(bg) / 255.0  # transform into a np array
            X[i] = cat

            Y[i, 0] = x1 / 128.0
            Y[i, 1] = y1 / 128.0
            Y[i, 2] = cat_x / 128.0
        yield X, Y


# plot the images
x, y = next(cat_gen())
plt.imshow(x[0])

Now, we can define a similar model, train it and examine its results:

Our trained model is capable of locating a cat in black background images:

In [ ]:
# transfer learning - load pre-trained vgg and replace its head
vgg = tf.keras.applications.VGG16(
    input_shape=[128, 128, 3], include_top=False, weights="imagenet"
)
x = Flatten()(vgg.output)
x = Dense(3, activation="sigmoid")(x)
model2 = Model(vgg.input, x)
model2.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.001))
# plot the model
plot_model(model2, "second_model.png", show_shapes=True)

# needs steps per epoch since the generator is infinite
model2.fit_generator(cat_gen(), steps_per_epoch=EPOCH_SIZE, epochs=5)

from matplotlib.patches import Rectangle


# given image and a label, plots the image + rectangle
def plot_pred(img, p):
    fig, ax = plt.subplots(1)
    ax.imshow(img)
    rect = Rectangle(
        xy=(p[0] * 128, p[1] * 128),
        width=p[2] * 128,
        height=p[2] * 128,
        linewidth=1,
        edgecolor="g",
        facecolor="none",
    )
    ax.add_patch(rect)
    plt.show()


# generate new image
x, _ = next(cat_gen())
# predict
pred = model2.predict(x)
# examine 1 image
im = x[0]
p = pred[0]
plot_pred(im, p)

### Final usecase

In this usecase, we will draw our cat over natural backgrounds. Similarly to the 2nd usecase, we will need to implement a new image generator - each image will consist of a random-sized cat in a random position. The cat will be drawn over a natural image, that will be its background. The natural image will be drawn randomly from a set of images.

# from PIL import Image

cat_pil = Image.open("cat.png")
cat_pil = cat_pil.resize((64, 64))
cat = np.asarray(cat_pil)


def natural_cat_gen(batch_size=64):
    # enable generating infinite amount of batches
    while True:
        # generate black images in the wanted size
        X = np.zeros((batch_size, 128, 128, 3))
        Y = np.zeros((batch_size, 3))
        # fill each image
        for i in range(batch_size):
            # resize the cat
            size = np.random.randint(32, 64)
            temp_cat = cat_pil.resize((size, size))
            cat = np.asarray(temp_cat) / 255.0
            cat_x, cat_y, _ = cat.shape
            # background image
            bg_name = f"bg{np.random.randint(1,4)}.jpg"
            bg = Image.open(bg_name)

            x1 = np.random.randint(1, 128 - cat_x)
            y1 = np.random.randint(1, 128 - cat_y)
            h = cat_x
            w = cat_y
            # draw the cat over the selected background image
            bg.paste(temp_cat, (x1, y1), mask=temp_cat)
            cat = np.asarray(bg) / 255.0
            X[i] = cat

            Y[i, 0] = x1 / 128.0
            Y[i, 1] = y1 / 128.0
            Y[i, 2] = cat_x / 128.0
        yield X, Y


# sanity check - plot the images
x, y = next(natural_cat_gen())
plt.imshow(x[0])

The natural_cat_gen() output will be similar to:

<a id='ObjectDetection'></a>
## Object Detection

In [ ]:
import os
import pathlib

import matplotlib
import matplotlib.pyplot as plt

import io
import scipy.misc
import numpy as np
from six import BytesIO
from PIL import Image, ImageDraw, ImageFont
from six.moves.urllib.request import urlopen

import tensorflow as tf
import tensorflow_hub as hub

tf.get_logger().setLevel("ERROR")

### Utilities

Run the following cell to create some utils that will be needed later:

- Helper method to load an image
- Map of Model Name to TF Hub handle
- List of tuples with Human Keypoints for the COCO 2017 dataset. This is needed for models with keypoints.

In [ ]:
# @title Run this!!


def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array.

    Puts image into numpy array to feed into tensorflow graph.
    Note that by convention we put it into a numpy array with shape
    (height, width, channels), where channels=3 for RGB.

    Args:
        path: the file path to the image

    Returns:
        uint8 numpy array with shape (img_height, img_width, 3)
    """
    image = None
    if path.startswith("http"):
        response = urlopen(path)
        image_data = response.read()
        image_data = BytesIO(image_data)
        image = Image.open(image_data)
    else:
        image_data = tf.io.gfile.GFile(path, "rb").read()
        image = Image.open(BytesIO(image_data))

    (im_width, im_height) = image.size
    return (
        np.array(image.getdata()).reshape((1, im_height, im_width, 3)).astype(np.uint8)
    )


ALL_MODELS = {
    "CenterNet HourGlass104 512x512": "https://tfhub.dev/tensorflow/centernet/hourglass_512x512/1",
    "CenterNet HourGlass104 Keypoints 512x512": "https://tfhub.dev/tensorflow/centernet/hourglass_512x512_kpts/1",
    "CenterNet HourGlass104 1024x1024": "https://tfhub.dev/tensorflow/centernet/hourglass_1024x1024/1",
    "CenterNet HourGlass104 Keypoints 1024x1024": "https://tfhub.dev/tensorflow/centernet/hourglass_1024x1024_kpts/1",
    "CenterNet Resnet50 V1 FPN 512x512": "https://tfhub.dev/tensorflow/centernet/resnet50v1_fpn_512x512/1",
    "CenterNet Resnet50 V1 FPN Keypoints 512x512": "https://tfhub.dev/tensorflow/centernet/resnet50v1_fpn_512x512_kpts/1",
    "CenterNet Resnet101 V1 FPN 512x512": "https://tfhub.dev/tensorflow/centernet/resnet101v1_fpn_512x512/1",
    "CenterNet Resnet50 V2 512x512": "https://tfhub.dev/tensorflow/centernet/resnet50v2_512x512/1",
    "CenterNet Resnet50 V2 Keypoints 512x512": "https://tfhub.dev/tensorflow/centernet/resnet50v2_512x512_kpts/1",
    "EfficientDet D0 512x512": "https://tfhub.dev/tensorflow/efficientdet/d0/1",
    "EfficientDet D1 640x640": "https://tfhub.dev/tensorflow/efficientdet/d1/1",
    "EfficientDet D2 768x768": "https://tfhub.dev/tensorflow/efficientdet/d2/1",
    "EfficientDet D3 896x896": "https://tfhub.dev/tensorflow/efficientdet/d3/1",
    "EfficientDet D4 1024x1024": "https://tfhub.dev/tensorflow/efficientdet/d4/1",
    "EfficientDet D5 1280x1280": "https://tfhub.dev/tensorflow/efficientdet/d5/1",
    "EfficientDet D6 1280x1280": "https://tfhub.dev/tensorflow/efficientdet/d6/1",
    "EfficientDet D7 1536x1536": "https://tfhub.dev/tensorflow/efficientdet/d7/1",
    "SSD MobileNet v2 320x320": "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2",
    "SSD MobileNet V1 FPN 640x640": "https://tfhub.dev/tensorflow/ssd_mobilenet_v1/fpn_640x640/1",
    "SSD MobileNet V2 FPNLite 320x320": "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/fpnlite_320x320/1",
    "SSD MobileNet V2 FPNLite 640x640": "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/fpnlite_640x640/1",
    "SSD ResNet50 V1 FPN 640x640 (RetinaNet50)": "https://tfhub.dev/tensorflow/retinanet/resnet50_v1_fpn_640x640/1",
    "SSD ResNet50 V1 FPN 1024x1024 (RetinaNet50)": "https://tfhub.dev/tensorflow/retinanet/resnet50_v1_fpn_1024x1024/1",
    "SSD ResNet101 V1 FPN 640x640 (RetinaNet101)": "https://tfhub.dev/tensorflow/retinanet/resnet101_v1_fpn_640x640/1",
    "SSD ResNet101 V1 FPN 1024x1024 (RetinaNet101)": "https://tfhub.dev/tensorflow/retinanet/resnet101_v1_fpn_1024x1024/1",
    "SSD ResNet152 V1 FPN 640x640 (RetinaNet152)": "https://tfhub.dev/tensorflow/retinanet/resnet152_v1_fpn_640x640/1",
    "SSD ResNet152 V1 FPN 1024x1024 (RetinaNet152)": "https://tfhub.dev/tensorflow/retinanet/resnet152_v1_fpn_1024x1024/1",
    "Faster R-CNN ResNet50 V1 640x640": "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1",
    "Faster R-CNN ResNet50 V1 1024x1024": "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_1024x1024/1",
    "Faster R-CNN ResNet50 V1 800x1333": "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_800x1333/1",
    "Faster R-CNN ResNet101 V1 640x640": "https://tfhub.dev/tensorflow/faster_rcnn/resnet101_v1_640x640/1",
    "Faster R-CNN ResNet101 V1 1024x1024": "https://tfhub.dev/tensorflow/faster_rcnn/resnet101_v1_1024x1024/1",
    "Faster R-CNN ResNet101 V1 800x1333": "https://tfhub.dev/tensorflow/faster_rcnn/resnet101_v1_800x1333/1",
    "Faster R-CNN ResNet152 V1 640x640": "https://tfhub.dev/tensorflow/faster_rcnn/resnet152_v1_640x640/1",
    "Faster R-CNN ResNet152 V1 1024x1024": "https://tfhub.dev/tensorflow/faster_rcnn/resnet152_v1_1024x1024/1",
    "Faster R-CNN ResNet152 V1 800x1333": "https://tfhub.dev/tensorflow/faster_rcnn/resnet152_v1_800x1333/1",
    "Faster R-CNN Inception ResNet V2 640x640": "https://tfhub.dev/tensorflow/faster_rcnn/inception_resnet_v2_640x640/1",
    "Faster R-CNN Inception ResNet V2 1024x1024": "https://tfhub.dev/tensorflow/faster_rcnn/inception_resnet_v2_1024x1024/1",
    "Mask R-CNN Inception ResNet V2 1024x1024": "https://tfhub.dev/tensorflow/mask_rcnn/inception_resnet_v2_1024x1024/1",
}

IMAGES_FOR_TEST = {
    "Beach": "models/research/object_detection/test_images/image2.jpg",
    "Dogs": "models/research/object_detection/test_images/image1.jpg",
    # By Heiko Gorski, Source: https://commons.wikimedia.org/wiki/File:Naxos_Taverna.jpg
    "Naxos Taverna": "https://upload.wikimedia.org/wikipedia/commons/6/60/Naxos_Taverna.jpg",
    # Source: https://commons.wikimedia.org/wiki/File:The_Coleoptera_of_the_British_islands_(Plate_125)_(8592917784).jpg
    "Beatles": "https://upload.wikimedia.org/wikipedia/commons/1/1b/The_Coleoptera_of_the_British_islands_%28Plate_125%29_%288592917784%29.jpg",
    # By Américo Toledano, Source: https://commons.wikimedia.org/wiki/File:Biblioteca_Maim%C3%B3nides,_Campus_Universitario_de_Rabanales_007.jpg
    "Phones": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg/1024px-Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg",
    # Source: https://commons.wikimedia.org/wiki/File:The_smaller_British_birds_(8053836633).jpg
    "Birds": "https://upload.wikimedia.org/wikipedia/commons/0/09/The_smaller_British_birds_%288053836633%29.jpg",
}

COCO17_HUMAN_POSE_KEYPOINTS = [
    (0, 1),
    (0, 2),
    (1, 3),
    (2, 4),
    (0, 5),
    (0, 6),
    (5, 7),
    (7, 9),
    (6, 8),
    (8, 10),
    (5, 6),
    (5, 11),
    (6, 12),
    (11, 12),
    (11, 13),
    (13, 15),
    (12, 14),
    (14, 16),
]

### Visualization tools

To visualize the images with the proper detected boxes, keypoints and segmentation, we will use the TensorFlow Object Detection API. To install it we will clone the repo.

### Clone the tensorflow models repository

```python
!git clone --depth 1 https://github.com/tensorflow/models
```

Installing the Object Detection API

<div class="alert alert-danger" role="alert">
    Note: This doesn't work on company devices due to the corporate firewall settings8 and tensorflow not supporting windows (gpus). Temp workaround, run on a server you control and run jupyter there with the following code (python 3.11, 3.12 not working as of time of writing);
    <br><br>
    On the server:<br>
    jupyter notebook --no-browser --port 1234<br><br>
    On your company device:<br>
    `ssh -NL 1234:localhost:1234 username@domain -p 22`
</div>

```bash
$ sudo apt install -y protobuf-compiler
$ cd models/research/
$ protoc object_detection/protos/*.proto --python_out=.
$ cp object_detection/packages/tf2/setup.py .
$ python -m pip install .
```

Now we can import the dependencies we will need later

In [ ]:
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as viz_utils
from object_detection.utils import ops as utils_ops

%matplotlib inline

Load label map data (for plotting).
Label maps correspond index numbers to category names, so that when our convolution network predicts 5, we know that this corresponds to airplane. Here we use internal utility functions, but anything that returns a dictionary mapping integers to appropriate string labels would be fine.

We are going, for simplicity, to load from the repository that we loaded the Object Detection API code

In [ ]:
PATH_TO_LABELS = "./models/research/object_detection/data/mscoco_label_map.pbtxt"
category_index = label_map_util.create_category_index_from_labelmap(
    PATH_TO_LABELS, use_display_name=True
)

Build a detection model and load pre-trained model weights
Here we will choose which Object Detection model we will use. Select the architecture and it will be loaded automatically. If you want to change the model to try other architectures later, just change the next cell and execute following ones.

Tip: if you want to read more details about the selected model, you can follow the link (model handle) and read additional documentation on TF Hub. After you select a model, we will print the handle to make it easier.

In [ ]:
model_display_name = "CenterNet HourGlass104 Keypoints 512x512"  # @param ['CenterNet HourGlass104 512x512','CenterNet HourGlass104 Keypoints 512x512','CenterNet HourGlass104 1024x1024','CenterNet HourGlass104 Keypoints 1024x1024','CenterNet Resnet50 V1 FPN 512x512','CenterNet Resnet50 V1 FPN Keypoints 512x512','CenterNet Resnet101 V1 FPN 512x512','CenterNet Resnet50 V2 512x512','CenterNet Resnet50 V2 Keypoints 512x512','EfficientDet D0 512x512','EfficientDet D1 640x640','EfficientDet D2 768x768','EfficientDet D3 896x896','EfficientDet D4 1024x1024','EfficientDet D5 1280x1280','EfficientDet D6 1280x1280','EfficientDet D7 1536x1536','SSD MobileNet v2 320x320','SSD MobileNet V1 FPN 640x640','SSD MobileNet V2 FPNLite 320x320','SSD MobileNet V2 FPNLite 640x640','SSD ResNet50 V1 FPN 640x640 (RetinaNet50)','SSD ResNet50 V1 FPN 1024x1024 (RetinaNet50)','SSD ResNet101 V1 FPN 640x640 (RetinaNet101)','SSD ResNet101 V1 FPN 1024x1024 (RetinaNet101)','SSD ResNet152 V1 FPN 640x640 (RetinaNet152)','SSD ResNet152 V1 FPN 1024x1024 (RetinaNet152)','Faster R-CNN ResNet50 V1 640x640','Faster R-CNN ResNet50 V1 1024x1024','Faster R-CNN ResNet50 V1 800x1333','Faster R-CNN ResNet101 V1 640x640','Faster R-CNN ResNet101 V1 1024x1024','Faster R-CNN ResNet101 V1 800x1333','Faster R-CNN ResNet152 V1 640x640','Faster R-CNN ResNet152 V1 1024x1024','Faster R-CNN ResNet152 V1 800x1333','Faster R-CNN Inception ResNet V2 640x640','Faster R-CNN Inception ResNet V2 1024x1024','Mask R-CNN Inception ResNet V2 1024x1024']
model_handle = ALL_MODELS[model_display_name]

print("Selected model:" + model_display_name)
print("Model Handle at TensorFlow Hub: {}".format(model_handle))

Loading the selected model from TensorFlow Hub
Here we just need the model handle that was selected and use the Tensorflow Hub library to load it to memory.

In [ ]:
print("loading model...")
hub_model = hub.load(model_handle)
print("model loaded!")

Loading an image
Let's try the model on a simple image. To help with this, we provide a list of test images.

Here are some simple things to try out if you are curious:

Try running inference on your own images, just upload them to colab and load the same way it's done in the cell below.
Modify some of the input images and see if detection still works. Some simple things to try out here include flipping the image horizontally, or converting to grayscale (note that we still expect the input image to have 3 channels).
Be careful: when using images with an alpha channel, the model expect 3 channels images and the alpha will count as a 4th.

Image Selection (don't forget to execute the cell!)

In [ ]:
selected_image = (
    "Beach"  # @param ['Beach', 'Dogs', 'Naxos Taverna', 'Beatles', 'Phones', 'Birds']
)
flip_image_horizontally = False
convert_image_to_grayscale = False

image_path = IMAGES_FOR_TEST[selected_image]
image_np = load_image_into_numpy_array(image_path)

# Flip horizontally
if flip_image_horizontally:
    image_np[0] = np.fliplr(image_np[0]).copy()

# Convert image to grayscale
if convert_image_to_grayscale:
    image_np[0] = np.tile(np.mean(image_np[0], 2, keepdims=True), (1, 1, 3)).astype(
        np.uint8
    )

plt.figure(figsize=(24, 32))
plt.imshow(image_np[0])
plt.show()

Doing the inference
To do the inference we just need to call our TF Hub loaded model.

Things you can try:

Print out result['detection_boxes'] and try to match the box locations to the boxes in the image. Notice that coordinates are given in normalized form (i.e., in the interval [0, 1]).
inspect other output keys present in the result. A full documentation can be seen on the models documentation page (pointing your browser to the model handle printed earlier)

In [ ]:
# running inference
results = hub_model(image_np)

# different object detection models have additional results
# all of them are explained in the documentation
result = {key: value.numpy() for key, value in results.items()}
print(result.keys())

Visualizing the results
Here is where we will need the TensorFlow Object Detection API to show the squares from the inference step (and the keypoints when available).

the full documentation of this method can be seen here

Here you can, for example, set min_score_thresh to other values (between 0 and 1) to allow more detections in or to filter out more detections.

In [ ]:
label_id_offset = 0
image_np_with_detections = image_np.copy()

# Use keypoints if available in detections
keypoints, keypoint_scores = None, None
if "detection_keypoints" in result:
    keypoints = result["detection_keypoints"][0]
    keypoint_scores = result["detection_keypoint_scores"][0]

viz_utils.visualize_boxes_and_labels_on_image_array(
    image_np_with_detections[0],
    result["detection_boxes"][0],
    (result["detection_classes"][0] + label_id_offset).astype(int),
    result["detection_scores"][0],
    category_index,
    use_normalized_coordinates=True,
    max_boxes_to_draw=200,
    min_score_thresh=0.30,
    agnostic_mode=False,
    keypoints=keypoints,
    keypoint_scores=keypoint_scores,
    keypoint_edges=COCO17_HUMAN_POSE_KEYPOINTS,
)

plt.figure(figsize=(24, 32))
plt.imshow(image_np_with_detections[0])
plt.show()

[Optional]
Among the available object detection models there's Mask R-CNN and the output of this model allows instance segmentation.

To visualize it we will use the same method we did before but adding an additional parameter: instance_masks=output_dict.get('detection_masks_reframed', None)

In [ ]:
# Handle models with masks:
image_np_with_mask = image_np.copy()

if "detection_masks" in result:
    # we need to convert np.arrays to tensors
    detection_masks = tf.convert_to_tensor(result["detection_masks"][0])
    detection_boxes = tf.convert_to_tensor(result["detection_boxes"][0])

    # Reframe the bbox mask to the image size.
    detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
        detection_masks, detection_boxes, image_np.shape[1], image_np.shape[2]
    )
    detection_masks_reframed = tf.cast(detection_masks_reframed > 0.5, tf.uint8)
    result["detection_masks_reframed"] = detection_masks_reframed.numpy()

viz_utils.visualize_boxes_and_labels_on_image_array(
    image_np_with_mask[0],
    result["detection_boxes"][0],
    (result["detection_classes"][0] + label_id_offset).astype(int),
    result["detection_scores"][0],
    category_index,
    use_normalized_coordinates=True,
    max_boxes_to_draw=200,
    min_score_thresh=0.30,
    agnostic_mode=False,
    instance_masks=result.get("detection_masks_reframed", None),
    line_thickness=8,
)

plt.figure(figsize=(24, 32))
plt.imshow(image_np_with_mask[0])
plt.show()

<a id='ImageSegmentation'></a>

## Image Segmentation

### What is image segmentation?

In an image classification task, the network assigns a label (or class) to each input image. However, suppose you want to know the shape of that object, which pixel belongs to which object, etc. In this case, you need to assign a class to each pixel of the image—this task is known as segmentation. A segmentation model returns much more detailed information about the image. Image segmentation has many applications in medical imaging, self-driving cars and satellite imaging, just to name a few.

This tutorial uses the [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/%7Evgg/data/pets/) (Parkhi et al, 2012). The dataset consists of images of 37 pet breeds, with 200 images per breed (~100 each in the training and test splits). Each image includes the corresponding labels, and pixel-wise masks. The masks are class-labels for each pixel. Each pixel is given one of three categories:

- Class 1: Pixel belonging to the pet.
- Class 2: Pixel bordering the pet.
- Class 3: None of the above/a surrounding pixel.

Required Libraries:

```bash
$ pip install git+https://github.com/tensorflow/examples.git
$ pip install -U keras
$ pip install -q tensorflow_datasets
$ pip install -q -U tensorflow-text tensorflow
```

In [ ]:
import numpy as np

import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
from tensorflow_examples.models.pix2pix import pix2pix

from IPython.display import clear_output
import matplotlib.pyplot as plt

### Download the Oxford-IIIT Pets dataset

The dataset is [available from TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/oxford_iiit_pet). The segmentation masks are included in version 3+.

In [ ]:
dataset, info = tfds.load("oxford_iiit_pet:3.*.*", with_info=True)

In addition, the image color values are normalized to the `[0, 1]` range. Finally, as mentioned above the pixels in the segmentation mask are labeled either {1, 2, 3}. For the sake of convenience, subtract 1 from the segmentation mask, resulting in labels that are : {0, 1, 2}.

In [ ]:
def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask -= 1
    return input_image, input_mask

In [ ]:
def load_image(datapoint):
    input_image = tf.image.resize(datapoint["image"], (128, 128))
    input_mask = tf.image.resize(
        datapoint["segmentation_mask"],
        (128, 128),
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR,
    )

    input_image, input_mask = normalize(input_image, input_mask)

    return input_image, input_mask

The dataset already contains the required training and test splits, so continue to use the same splits:

In [ ]:
TRAIN_LENGTH = info.splits["train"].num_examples
BATCH_SIZE = 64
BUFFER_SIZE = 1000
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

In [ ]:
train_images = dataset["train"].map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_images = dataset["test"].map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

The following class performs a simple augmentation by randomly-flipping an image. Go to the [Image augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation) tutorial to learn more.

In [ ]:
class Augment(tf.keras.layers.Layer):
    def __init__(self, seed=42):
        super().__init__()
        # both use the same seed, so they'll make the same random changes.
        self.augment_inputs = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)
        self.augment_labels = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)

    def call(self, inputs, labels):
        inputs = self.augment_inputs(inputs)
        labels = self.augment_labels(labels)
        return inputs, labels

## Context Aware Chatbots

### Installing Haystack

In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

In [ ]:
document_store = InMemoryDocumentStore()

In [ ]:
from datasets import load_dataset
from haystack import Document

In [ ]:
dataset = load_dataset("bilgeyucel/seven-wonders", split="train")
docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in dataset]

In [ ]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

# Appendix B - Preprocessing Images

## Support Functions

In [ ]:
from matplotlib import pyplot as plt
from pathlib import Path
import pytesseract
import numpy as np
import cv2
import os

# Reading original `PLANINDX.TXT`
planset_path = Path(os.getcwd()).parent / "res/sample_plan_tiff"

In [ ]:
x = 5

In [ ]:
tiff_list = [planset_path / file for file in os.listdir(planset_path)]

## Loading an Image to Read

In [ ]:
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = (
    "C:\\Users\\dparks1\\AppData\\Local\\Programs\\Tesseract-OCR\\tesseract.exe"
)

In [ ]:
# Show the first image being read
im = Image.open(tiff_list[2])

# im

In [ ]:
img = cv2.imread(str(tiff_list[2]))

In [ ]:
# https://stackoverflow.com/questions/28816046/
# displaying-different-images-with-actual-size-in-matplotlib-subplot
def display(im_path):
    dpi = 80
    im_data = plt.imread(im_path)

    height, width = im_data.shape[:2]

    # What size does the figure need to be in inches to fit the image?
    figsize = width / float(dpi), height / float(dpi)

    # Create a figure of the right size with one axes that takes up the full figure
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0, 0, 1, 1])

    # Hide spines, ticks, etc.
    ax.axis("off")

    # Display the image.
    ax.imshow(im_data, cmap="gray")

    plt.show()

## Inverted Image

Inverts color spectrum of image, no longer necessary in tesseract > 4.0

In [ ]:
inverted_image = cv2.bitwise_not(img)
cv2.imwrite("output/temp/inverted.jpg", inverted_image)

In [ ]:
# display('output/temp/inverted.jpg')

## Rescaling

## Binarization

In [ ]:
def grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [ ]:
gray_image = grayscale(img)
cv2.imwrite("output/temp/gray.jpg", gray_image)

In [ ]:
# display('output/temp/gray.jpg')

In [ ]:
thresh, im_bw = cv2.threshold(gray_image, 200, 250, cv2.THRESH_BINARY)
cv2.imwrite("output/temp/bw_image.jpg", im_bw)

In [ ]:
# display('output/temp/bw_image.jpg')

## Noise Removal

In [ ]:
def noise_removal(image):
    kernel = np.ones((1, 1), np.uint8)
    image = cv2.dilate(image, kernel, iterations=1)
    kernel = np.ones((1, 1), np.uint8)
    image = cv2.erode(image, kernel, iterations=1)
    image = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)
    image = cv2.medianBlur(image, 3)
    return image

In [ ]:
no_noise = noise_removal(im_bw)
cv2.imwrite("output/temp/no_noise.jpg", no_noise)

In [ ]:
# display('output/temp/no_noise.jpg')

## Dilation and Erosion

In [ ]:
def thin_font(image):
    image = cv2.bitwise_not(image)
    kernel = np.ones((2, 2), np.uint8)
    image = cv2.erode(image, kernel, iterations=2)
    image = cv2.bitwise_not(image)
    return image

In [ ]:
eroded_image = thin_font(no_noise)
cv2.imwrite("output/temp/eroded_imge.jpg", eroded_image)

In [ ]:
# display('output/temp/eroded_imge.jpg')

In [ ]:
def thick_font(image):
    image = cv2.bitwise_not(image)
    kernel = np.ones((2, 2), np.uint8)
    image = cv2.dilate(image, kernel, iterations=5)
    image = cv2.bitwise_not(image)

    return image

In [ ]:
dilated_image = thick_font(no_noise)
cv2.imwrite("output/temp/dilated.jpg", dilated_image)

In [ ]:
# display('output/temp/dilated.jpg')

## Rotation / Deskewing

- Didn't work with planset files

In [ ]:
# Show the first image being read
im2 = cv2.imread(str(tiff_list[1]))

In [ ]:
# https://becominghuman.ai/how-to-automatically-deskew-straighten-a-text-image-using-opencv-a0c30aed83df


def getSkewAngle(cvImage) -> float:
    # Prep image, copy, convert to gray scale, blur, and threshold
    newImage = cvImage.copy()
    gray = cv2.cvtColor(newImage, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Apply dilate to merge text into meaningful lines/paragraphs.
    # Use larger kernel on X axis to merge characters into single line, cancelling out any spaces.
    # But use smaller kernel on Y axis to separate between different blocks of text
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
    dilate = cv2.dilate(thresh, kernel, iterations=2)

    # Find all contours
    contours, hierarchy = cv2.findContours(
        dilate, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
    )
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    for c in contours:
        rect = cv2.boundingRect(c)
        x, y, w, h = rect
        cv2.rectangle(newImage, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Find largest contour and surround in min area box
    largestContour = contours[0]
    print(len(contours))
    minAreaRect = cv2.minAreaRect(largestContour)
    cv2.imwrite("temp/boxes.jpg", newImage)
    # Determine the angle. Convert it to the value that was originally used to obtain skewed image
    angle = minAreaRect[-1]
    if angle < -45:
        angle = 90 + angle
    return -1.0 * angle


# Rotate the image around its center
def rotateImage(cvImage, angle: float):
    newImage = cvImage.copy()
    (h, w) = newImage.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    newImage = cv2.warpAffine(
        newImage, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    return newImage


# Deskew image
def deskew(cvImage):
    angle = getSkewAngle(cvImage)
    return rotateImage(cvImage, -1.0 * angle)

In [ ]:
fixed = deskew(im2)
cv2.imwrite("output/temp/rotated_fixed.jpg", fixed)

In [ ]:
# Fails because the image contains a border
angle = getSkewAngle(im2)
print(angle)

## Remove Borders

- Didn't work with planset files

In [ ]:
# display('output/temp/no_noise.jpg')

In [ ]:
im = np.array(Image.open("output/temp/no_noise.jpg").convert("L"))


def remove_borders(image):
    # Load image and greyscale it
    # im = cv2.bitwise_not(image)

    # Normalize and threshold image
    im = cv2.normalize(image, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    res, im = cv2.threshold(im, 64, 255, cv2.THRESH_BINARY)

    # Fill everything that is the same colour (black) as top-left corner with white
    cv2.floodFill(im, None, (0, 0), 255)

    # Fill everything that is the same colour (white) as top-left corner with black
    # cv2.floodFill(im, None, (0,0), 0)

    return im

In [ ]:
no_borders = remove_borders(im)
cv2.imwrite("output/temp/no_borders.jpg", no_borders)
# display('output/temp/no_borders.jpg')

# 